In [ ]:
import os

os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"   
os.environ["CUDA_VISIBLE_DEVICES"]='0'

In [ ]:
import torch
import random
import numpy as np

def set_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
set_seeds(42)

In [ ]:
import matplotlib.pyplot as plt
def show_images(images, scores, test_artist, train_artists):
    n: int = len(images)
    f = plt.figure(figsize=(16, 2))
    for i in range(n):
        # Debug, plot figure
        ax = f.add_subplot(1, n, i + 1)
        if i==0:
            pass
            ax.title.set_text(test_artist)
        else:
            ax.title.set_text(str(np.round(scores[i-1], 4))+'\n'+train_artists[i-1])
            ax.axis('off')
        if images[i]==None:
            pass
        else:
            plt.imshow(images[i])

    plt.show(block=True)

In [ ]:
from datasets import load_dataset

In [ ]:
import pickle

In [ ]:
with open('../../data/indices/5000-0.5/idx-train.pkl', 'rb')  as handle:
    idx_train = pickle.load(handle)
len(idx_train)   

In [ ]:
# with open('../../data/indices/5000-0.5/idx-val.pkl', 'rb')  as handle:
#     idx_val = pickle.load(handle)
# len(idx_val)

In [ ]:
import pandas as pd

df = pd.read_csv('/home/yourname/.cache/kagglehub/datasets/alexanderliao/artbench10/versions/2/ArtBench-10.csv')
#df = pd.read_csv('../../../../codes/artbench/ArtBench-10.csv')
df.head()

In [ ]:
df['path'] = df.apply(lambda x: "/home/yourname/neurips/artbench-merged/{}/{}".format(x['label'], x['name']), axis=1)
#df['path'] = df.apply(lambda x: "../../../../codes/artbench/data/artbench-10-imagefolder/{}/{}".format(x['label'], x['name']), axis=1)
df.head()

In [ ]:
from datasets import Dataset, load_dataset, Image

train_dataset = Dataset.from_dict({"image": df.loc[idx_train]['path'].tolist(),
                                   "label": df.loc[idx_train]['label'].tolist(),
                                  }).cast_column("image", Image())
train_dataset[0]["image"]

In [ ]:
import pandas as pd
df = pd.DataFrame()
df['label'] = ['ukiyo_e']*500+['post_impressionism']*500
df['path'] = ['{}/{}.png'.format('../../saved/5000-0.5/gen', i) for i in range(1000)]

from datasets import DatasetDict, Dataset, load_dataset, Image
dataset = DatasetDict({
"train": Dataset.from_dict({
    "image": df['path'].tolist(),
    "label": df['label'].tolist(),
}).cast_column("image", Image()),})
val_dataset = dataset["train"]
val_dataset[0]["image"]

In [ ]:
import numpy as np
import torch
from pkg_resources import packaging

print("Torch version:", torch.__version__)

Open Clip

In [ ]:
import open_clip

model, _, preprocess = open_clip.create_model_and_transforms('ViT-g-14', pretrained='laion2b_s34b_b88k')

model.cuda().eval()

input_resolution = model.visual.image_size
print(f"Model: ViT-g-14 | Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Input resolution: {input_resolution}")

context_length = model.context_length
vocab_size = model.vocab_size


In [ ]:
preprocess

In [ ]:
train_features = []
for i in range(0, len(train_dataset), 32):
    batch = train_dataset[i:i+32]['image']
    batch = [preprocess(b) for b in batch]
    batch = torch.tensor(np.stack(batch)).cuda()
    with torch.no_grad():
        image_features = model.encode_image(batch).float()
    print(image_features.size())
    train_features.append(image_features.cpu().numpy())

In [ ]:
train_features_array = np.vstack(train_features)
train_features_array.shape

In [ ]:
val_features = []
for i in range(0, len(val_dataset), 32):
    batch = val_dataset[i:i+32]['image']
    batch = [preprocess(b) for b in batch]
    batch = torch.tensor(np.stack(batch)).cuda()
    with torch.no_grad():
        image_features = model.encode_image(batch).float()
    print(image_features.size())
    val_features.append(image_features.cpu().numpy())

In [ ]:
val_features_array = np.vstack(val_features)
val_features_array.shape

load mpl

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ProjectionHead(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim)
        )
    def forward(self, x):
        x = self.mlp(x)
        return F.normalize(x, p=2, dim=1)

class MultiModalAlignmentModel(nn.Module):
    def __init__(self, img_in_dim=1024, graph_in_dim=512, shared_dim=512):
        super().__init__()
        self.image_proj = ProjectionHead(input_dim=img_in_dim, hidden_dim=512, output_dim=shared_dim)
        self.graph_proj = ProjectionHead(input_dim=graph_in_dim, hidden_dim=512, output_dim=shared_dim)

    def forward(self, img_embeds, graph_embeds):
        return self.image_proj(img_embeds), self.graph_proj(graph_embeds)

# Instantiate and Load the Weights
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MultiModalAlignmentModel(img_in_dim=1024, graph_in_dim=512, shared_dim=512).to(device)

# Load the saved .pth file (make sure the path is correct)
model.load_state_dict(torch.load('/home/yourname/contrastive_learning/projection_heads_v1/new_multimodal_projection_heads_vitg14_node2vec.pth', map_location=device))
#model.load_state_dict(torch.load('/home/yourname/contrastive_learning/projection_heads_v1/multimodal_projection_heads_vitg14_hashgnn.pth', map_location=device))

model.eval()
print("Trained MLPs successfully loaded!")

In [ ]:
import torch
import numpy as np
from sklearn.cluster import KMeans


print(f"Old train shape: {train_features_array.shape}")
print(f"Old val shape: {val_features_array.shape}")

with torch.no_grad(): 
    train_tensor = torch.tensor(train_features_array, dtype=torch.float32).to(device)
    val_tensor = torch.tensor(val_features_array, dtype=torch.float32).to(device)
    
    shared_train_features = model.image_proj(train_tensor)
    shared_val_features = model.image_proj(val_tensor)
    
    shared_train_array = shared_train_features.cpu().numpy()
    shared_val_array = shared_val_features.cpu().numpy()

print(f"New shared train shape: {shared_train_array.shape}")
print(f"New shared val shape: {shared_val_array.shape}")


def normalize_l2(arrays):
    return arrays / (np.linalg.norm(arrays, axis=1, keepdims=True) + 1e-8)

val_v_norm = normalize_l2(val_features_array)
train_v_norm = normalize_l2(train_features_array)

shared_val_norm = normalize_l2(shared_val_array)
shared_train_norm = normalize_l2(shared_train_array)


sim_visual = np.maximum(val_v_norm.dot(train_v_norm.T), 0)
sim_domain = np.maximum(shared_val_norm.dot(shared_train_norm.T), 0)

p1, p5, p10 = 1, 5, 10

a, b = 0.7, 0.3 

# Visual Only
v_pow_1  = np.power(sim_visual, p1)
v_pow_5  = np.power(sim_visual, p5)
v_pow_10 = np.power(sim_visual, p10)

# Domain Only
d_pow_1  = np.power(sim_domain, p1)
d_pow_5  = np.power(sim_domain, p5)
d_pow_10 = np.power(sim_domain, p10)

# Hybrid (Additive after power)
h_pow_1  = (a * v_pow_1)  + (b * d_pow_1)
h_pow_5  = (a * v_pow_5)  + (b * d_pow_5)
h_pow_10 = (a * v_pow_10) + (b * d_pow_10)


num_clusters = 10  

print("Computing K-Means Priors...")
kmeans_v = KMeans(n_clusters=num_clusters, random_state=42, n_init=10).fit(train_v_norm)
v_distances = np.linalg.norm(train_v_norm - kmeans_v.cluster_centers_[kmeans_v.labels_], axis=1)

kmeans_d = KMeans(n_clusters=num_clusters, random_state=42, n_init=10).fit(shared_train_norm)
d_distances = np.linalg.norm(shared_train_norm - kmeans_d.cluster_centers_[kmeans_d.labels_], axis=1)

cluster_v_prior = 0.5 + (v_distances - v_distances.min()) / (v_distances.max() - v_distances.min() + 1e-8)
cluster_d_prior = 0.5 + (d_distances - d_distances.min()) / (d_distances.max() - d_distances.min() + 1e-8)

cluster_v_prior = cluster_v_prior.reshape(1, -1)
cluster_d_prior = cluster_d_prior.reshape(1, -1)

combined_prior = (0.5 * cluster_v_prior) + (0.5 * cluster_d_prior)


# Likelihood * Prior
v_pow_10_with_prior = v_pow_10 * cluster_v_prior   
d_pow_10_with_prior = d_pow_10 * cluster_d_prior
h_pow_10_with_prior = h_pow_10 * combined_prior


scores_list = [
    # --- Group 1: Visual Likelihood ---
#    v_pow_1,                   # 1. Visual (Linear)
#    v_pow_5,                   # 2. Visual (p=5)
#    v_pow_10,                  # 3. Visual (p=10)
    
    # --- Group 2: Domain Likelihood ---
#    d_pow_1,                   # 4. Domain (Linear)
#    d_pow_5,                   # 5. Domain (p=5)
#    d_pow_10,                  # 6. Domain (p=10)
    
    # --- Group 3: Hybrid Likelihood ---
#    h_pow_1,                   # 7. Hybrid (Linear)
#    h_pow_5,                   # 8. Hybrid (p=5)
#    h_pow_10,                  # 9. Hybrid (p=10)
    
    # --- Group 4: Full Bayesian Framework (Likelihood * Prior) ---
#    v_pow_10_with_prior,       # 10. Visual p=10 + Visual Prior
    d_pow_10_with_prior,       # 11. Domain p=10 + Domain Prior
    h_pow_10_with_prior        # 12. Hybrid p=10 + Combined Prior
]

In [ ]:
import numpy as np

ranks_v = np.argsort(np.argsort(-v_pow_10_with_prior, axis=1), axis=1) + 1
ranks_d = np.argsort(np.argsort(-d_pow_10_with_prior, axis=1), axis=1) + 1


k_standard = 60 
pure_rrf_60 = (1.0 / (k_standard + ranks_v)) + (1.0 / (k_standard + ranks_d))

k_steep = 10
pure_rrf_10 = (1.0 / (k_steep + ranks_v)) + (1.0 / (k_steep + ranks_d))

scores_list = [
#    v_pow_10_with_prior,        
#    final_fully_priored_boost,  
    pure_rrf_60,                
    pure_rrf_10                
]


In [ ]:
print("Computing IR-Inspired Rank Fusions...")


ranks_v = np.argsort(np.argsort(-v_pow_10_with_prior, axis=1), axis=1) + 1
ranks_d = np.argsort(np.argsort(-d_pow_10_with_prior, axis=1), axis=1) + 1

W_custom = 1.0 
k_custom = 5   
final_fully_priored_boost = v_pow_10_with_prior * (1.0 + (W_custom / (k_custom + ranks_d)))

k_standard = 60 
pure_rrf_60 = (1.0 / (k_standard + ranks_v)) + (1.0 / (k_standard + ranks_d))

k_steep = 10
pure_rrf_10 = (1.0 / (k_steep + ranks_v)) + (1.0 / (k_steep + ranks_d))

scores_list = [
#    v_pow_10_with_prior,        
    final_fully_priored_boost,   
    pure_rrf_60,                
#    pure_rrf_10                 
]

In [ ]:
with open('./gen_clip.pkl', 'wb') as handle:
    pickle.dump(scores_list, handle)

In [ ]:
my_list = [
    0,1,2,3,
    4,5,6,7,
    8,9,10,11,
    12,13,14,15,
    16,17,18,19,
    20,21,22,23,
    24,25,26,27,
    28,29,30,31,
    32,33,34,35,
    36,37,38,39,
    40,41,42,43,
    44,45,46,47,
    48,49,50,51,
    52,53,54,55,
    56,57,58,59,
    60,61,62,63,
          ]

In [ ]:
loss_array_list = []

for i in my_list:
    for seed in [
        0,
                 1,
                 2,
                 # 3,
                 # 4,
                ]:
        for e_seed in [
            0, 
                       1, 
                       2
                      ]:
            with open('../../saved/5000-0.5/lds-val/sd-lora-sub-{}-{}/e-{}-gen.pkl'.format(i, seed, e_seed), 'rb')  as handle:
                loss_list = pickle.load(handle)
            margins = np.concatenate(loss_list, axis=-1) # -logp
            ####
            if (seed==0) and (e_seed)==0:
                loss_array = margins
            else:
                loss_array += margins
            
    loss_array = loss_array/(3*3)
    
    loss_array_list.append(loss_array)
lds_loss_array = np.stack(loss_array_list)
lds_loss_array.shape

In [ ]:
mask_array_list = []

for i in my_list:
    # print(i)
    with open('../../data/indices/5000-0.5/lds-val/sub-idx-{}.pkl'.format(i), 'rb')  as handle:
        sub_idx_train = pickle.load(handle)
    # print(len(sub_idx_train))
    mask_array = np.in1d(idx_train, sub_idx_train)
        
    mask_array_list.append(mask_array)
    
lds_mask_array = np.stack(mask_array_list)
lds_mask_array.shape

In [ ]:
lds_testset_correctness = lds_loss_array.mean(axis=1)
lds_testset_correctness.shape

In [ ]:
for j in range(4):
    plt.plot(lds_testset_correctness[:, j], color="C{}".format(j))
    # break
# plt.ylim(0.15, 0.2)

In [ ]:
# compute lds
from scipy.stats import spearmanr, pearsonr
####
# k = 48
# margins = lds_testset_correctness[k:]
# infl_est_ = _masked_dot(lds_testset_correctness[:k], lds_mask_array[:k]) - _masked_dot(lds_testset_correctness[:k], ~lds_mask_array[:k])
# # infl_est_ = _masked_dot(lds_testset_correctness[:], lds_mask_array[:]) - _masked_dot(lds_testset_correctness[:], ~lds_mask_array[:])
# preds = lds_mask_array[k:] @ infl_est_.T

margins = lds_testset_correctness
np.random.seed(0)
infl_est_ = -np.random.rand(1000, 5000)
# infl_est_ = -tmp
preds = lds_mask_array @ infl_est_.T
####
rs = []
ps = []

for ind in range(1000):
    r, p = spearmanr(preds[:, ind], margins[:, ind])
    # r, p = pearsonr(preds[:, ind], margins[:, ind])
    rs.append(r)
    ps.append(p)
    
rs, ps = np.array(rs), np.array(ps)
print(f'Correlation: {rs.mean():.3f} (avg p value {ps.mean():.6f})')

In [ ]:
# compute lds
from scipy.stats import spearmanr, pearsonr
####
# k = 48
# margins = lds_testset_correctness[k:]
# infl_est_ = _masked_dot(lds_testset_correctness[:k], lds_mask_array[:k]) - _masked_dot(lds_testset_correctness[:k], ~lds_mask_array[:k])
# # infl_est_ = _masked_dot(lds_testset_correctness[:], lds_mask_array[:]) - _masked_dot(lds_testset_correctness[:], ~lds_mask_array[:])
# preds = lds_mask_array[k:] @ infl_est_.T

margins = lds_testset_correctness
np.random.seed(1)
infl_est_ = -np.random.rand(1000, 5000)
# infl_est_ = -tmp
preds = lds_mask_array @ infl_est_.T
####
rs = []
ps = []

for ind in range(1000):
    r, p = spearmanr(preds[:, ind], margins[:, ind])
    # r, p = pearsonr(preds[:, ind], margins[:, ind])
    rs.append(r)
    ps.append(p)
    
rs, ps = np.array(rs), np.array(ps)
print(f'Correlation: {rs.mean():.3f} (avg p value {ps.mean():.6f})')

In [ ]:
# compute lds
from scipy.stats import spearmanr, pearsonr
####
# k = 48
# margins = lds_testset_correctness[k:]
# infl_est_ = _masked_dot(lds_testset_correctness[:k], lds_mask_array[:k]) - _masked_dot(lds_testset_correctness[:k], ~lds_mask_array[:k])
# # infl_est_ = _masked_dot(lds_testset_correctness[:], lds_mask_array[:]) - _masked_dot(lds_testset_correctness[:], ~lds_mask_array[:])
# preds = lds_mask_array[k:] @ infl_est_.T

margins = lds_testset_correctness
np.random.seed(2)
infl_est_ = -np.random.rand(1000, 5000)
# infl_est_ = -tmp
preds = lds_mask_array @ infl_est_.T
####
rs = []
ps = []

for ind in range(1000):
    r, p = spearmanr(preds[:, ind], margins[:, ind])
    # r, p = pearsonr(preds[:, ind], margins[:, ind])
    rs.append(r)
    ps.append(p)
    
rs, ps = np.array(rs), np.array(ps)
print(f'Correlation: {rs.mean():.3f} (avg p value {ps.mean():.6f})')

In [ ]:
(0.005 + -0.001 + -0.004)/3.0

In [ ]:
# compute lds
from scipy.stats import spearmanr, pearsonr
####
# k = 48
# margins = lds_testset_correctness[k:]
# infl_est_ = _masked_dot(lds_testset_correctness[:k], lds_mask_array[:k]) - _masked_dot(lds_testset_correctness[:k], ~lds_mask_array[:k])
# # infl_est_ = _masked_dot(lds_testset_correctness[:], lds_mask_array[:]) - _masked_dot(lds_testset_correctness[:], ~lds_mask_array[:])
# preds = lds_mask_array[k:] @ infl_est_.T

margins = lds_testset_correctness
infl_est_ = -scores_list[0]
# infl_est_ = -tmp
preds = lds_mask_array @ infl_est_.T
####
rs = []
ps = []

for ind in range(1000):
    r, p = spearmanr(preds[:, ind], margins[:, ind])
    # r, p = pearsonr(preds[:, ind], margins[:, ind])
    rs.append(r)
    ps.append(p)
    
rs, ps = np.array(rs), np.array(ps)
print(f'Correlation: {rs.mean():.3f} (avg p value {ps.mean():.6f})')

In [ ]:
my_data = {
    'margins': margins[:, 0],
    'preds': preds[:, 0]
}

In [ ]:
import seaborn as sns
sns.jointplot(data=my_data, x="margins", y="preds", kind="reg")

In [ ]:
# compute lds
from scipy.stats import spearmanr, pearsonr
####
# k = 48
# margins = lds_testset_correctness[k:]
# infl_est_ = _masked_dot(lds_testset_correctness[:k], lds_mask_array[:k]) - _masked_dot(lds_testset_correctness[:k], ~lds_mask_array[:k])
# # infl_est_ = _masked_dot(lds_testset_correctness[:], lds_mask_array[:]) - _masked_dot(lds_testset_correctness[:], ~lds_mask_array[:])
# preds = lds_mask_array[k:] @ infl_est_.T

margins = lds_testset_correctness
infl_est_ = -scores_list[1]
# infl_est_ = -tmp
preds = lds_mask_array @ infl_est_.T
####
rs = []
ps = []

for ind in range(1000):
    r, p = spearmanr(preds[:, ind], margins[:, ind])
    # r, p = pearsonr(preds[:, ind], margins[:, ind])
    rs.append(r)
    ps.append(p)
    
rs, ps = np.array(rs), np.array(ps)
print(f'Correlation: {rs.mean():.3f} (avg p value {ps.mean():.6f})')

In [ ]:
my_data = {
    'margins': margins[:, 0],
    'preds': preds[:, 0]
}

In [ ]:
import seaborn as sns
sns.jointplot(data=my_data, x="margins", y="preds", kind="reg")

In [ ]:
scores = scores_list[1]

In [ ]:
i = 0

In [ ]:
D = -scores[i]
D.shape

In [ ]:
plt.plot(sorted(D))
# plt.axhline(y=0, c='red')

In [ ]:
topK = np.arange(5000)[D.argsort()[0:5]]
topK

In [ ]:
plot_images = []
plot_images.append(val_dataset[i]['image'])
for idx in topK:
    plot_images.append(train_dataset[int(idx)]['image'])

In [ ]:
val_artist = ''
val_artist

In [ ]:
train_artist = []
for k in topK:
    tmp_artist = ''
    train_artist.append(tmp_artist)
train_artist   

In [ ]:
# full
show_images(plot_images, D[D.argsort()[0:5]], val_artist, train_artist)

In [ ]:
#done